In [1]:
import pandas as pd
import numpy as np

In [3]:
print("1. Loading Olist Data...")
orders = pd.read_csv('data\olist_orders_dataset.csv')
items = pd.read_csv('data\olist_order_items_dataset.csv')
customers = pd.read_csv('data\olist_customers_dataset.csv')
geolocation = pd.read_csv('data\olist_geolocation_dataset.csv')

1. Loading Olist Data...


In [4]:
print("2. Engineering Delivery Time...")
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['Delivery_Time_Mins'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / 60.0
orders = orders.dropna(subset=['Delivery_Time_Mins'])

2. Engineering Delivery Time...


In [ ]:
print("3. Calculating Total Order Values...")
items['total_item_value'] = items['price'] + items['freight_value']
order_values = items.groupby('order_id')['total_item_value'].sum().reset_index()
order_values.rename(columns={'total_item_value': 'Order_Value_INR'}, inplace=True)

3. Calculating Total Order Values...


In [6]:
print("4. Cleaning Geolocation Coordinates...")
geo_clean = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

4. Cleaning Geolocation Coordinates...


In [ ]:
print("5. Merging the Tables (SQL-style Joins)...")
df_merged = orders[['order_id', 'customer_id', 'Delivery_Time_Mins']].merge(
    customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id', how='inner'
)
df_merged = df_merged.merge(order_values, on='order_id', how='inner')
df_merged = df_merged.merge(
    geo_clean, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner'
)

5. Merging the Tables (SQL-style Joins)...


In [10]:
print("6. Finalizing Dataset for ML Pipeline...")
final_columns = ['order_id', 'geolocation_lat', 'geolocation_lng', 'Order_Value_INR', 'Delivery_Time_Mins']
real_delivery_df = df_merged[final_columns].copy()
real_delivery_df.columns = ['order_id', 'Latitude', 'Longitude', 'Order_Value_INR', 'Delivery_Time_Mins']
real_delivery_df = real_delivery_df[
    (real_delivery_df['Delivery_Time_Mins'] > 0) & 
    (real_delivery_df['Order_Value_INR'] > 0)
]

6. Finalizing Dataset for ML Pipeline...


In [12]:
# 7. Save the merged real-world data to a CSV file!
real_delivery_df.to_csv('real_delivery_data.csv', index=False)
print("Saved successfully to 'real_delivery_data.csv'")

Saved successfully to 'real_delivery_data.csv'
